# RAG 개요, LLM 한계와 "외부 지식" 보강
- LLM 은 학습 시점 이후의 정보를 모르고, 우리 회사 내부 문서 같은 비공개 정보도 모름
- 매번 시스템 프롬프트에 모든 정보를 넣는 건 토큰 한계 때문에 불가능
- **RAG (Retrieval-Augmented Generation)** = "질문에 맞는 정보를 외부에서 찾아 와서 LLM 에 같이 주입"

## RAG 시스템 구축 순서

1. **Load** 문서 가져오기 (PDF, MD, HTML, DB)
2. **Split** 청크로 자르기 (LLM 컨텍스트에 맞게)
3. **Embed** 각 청크를 벡터로 (의미 검색용)
4. **Store** 벡터 DB 에 저장
5. **Retrieve & Generate** 질문을 벡터로 → 유사한 청크 검색 → LLM 답변


## 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.


In [1]:
# 필요한 라이브러리 설치
# uv add langchain-openai numpy scikit-learn

## (2) 라이브러리 Import

이번 실습에서 사용하는 핵심 객체는 다음과 같습니다.

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

## 2. "비슷한 문장" 을 컴퓨터가 어떻게 알까
- LLM 은 텍스트를 **벡터(숫자 배열)** 로 바꿔서 비교함
- 의미가 비슷하면 벡터 거리가 가까움

In [3]:
from langchain_openai import OpenAIEmbeddings
import numpy as np

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

sentences = [
    '마법사는 마나를 다루는 직업이다.',
    '위자드는 마법 에너지로 싸우는 캐릭터다.',
    '오늘 점심에 김치찌개를 먹어야지'
]

vectors = embeddings.embed_documents(sentences)
print(f'벡터 차원: {len(vectors[0])}')          # 

벡터 차원: 1536


## 3. 코사인 유사도, "두 벡터가 얼마나 같은 방향을 가리키나"

In [4]:
def cosine_sim(a, b):
    """코사인 유사도. 1 = 같은 방향(=의미가 같음), 0 = 무관, -1 = 정반대."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print(f"[마법사 vs 위자드]  {cosine_sim(vectors[0], vectors[1]):.3f}")
print(f"[마법사 vs 김치찌개]  {cosine_sim(vectors[0], vectors[2]):.3f}")
print(f"[위자드 vs 김치찌개]  {cosine_sim(vectors[1], vectors[2]):.3f}")

[마법사 vs 위자드]  0.314
[마법사 vs 김치찌개]  0.114
[위자드 vs 김치찌개]  0.091


## 4. 질문도 같은 방식으로 벡터화

In [5]:
docs = [
    "신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사내 시스템 접근 권한이 제한될 수 있다.",
    "법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다.",
    "개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연락처는 마스킹 대상에 포함된다.",
    "장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비스 복구 후 24시간 이내에 작성해야 한다.",
    "재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.",
]

doc_vectors = embeddings.embed_documents(docs)

In [6]:
question = '법인카드 영수증은 언제까지 제출해야 하나요?'
q_vector = embeddings.embed_query(question)

In [7]:
# 질문 vs 각 문서 유사도
scores = [cosine_sim(q_vector, dv) for dv in doc_vectors]
ranked = sorted(zip(scores, docs), reverse=True)

print(f"질문: {question}\n")
print("=== 유사도 순위 ===")
for score, doc in ranked:
    print(f"  {score:.3f}  {doc}")

질문: 법인카드 영수증은 언제까지 제출해야 하나요?

=== 유사도 순위 ===
  0.653  법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다.
  0.228  개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연락처는 마스킹 대상에 포함된다.
  0.227  재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.
  0.187  신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사내 시스템 접근 권한이 제한될 수 있다.
  0.099  장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비스 복구 후 24시간 이내에 작성해야 한다.


> 위 결과에서 상위 1~2 개 문서를 LLM 에 "참고 자료" 로 같이 넣으면 그게 가장 단순한 RAG.

## 5. 임베딩 모델 비교 표 (2026-06)

| 모델 | 기본 차원 | 한국어 품질 | 가격(1M input tokens, standard) |
|---|---|---|---|
| `text-embedding-3-small` | 1536 (조절 가능) | 좋음 | 0.02 달러 |
| `text-embedding-3-large` | 3072 (조절 가능) | 우수 | 0.13 달러 |
| `bge-m3` (오픈) | 1024 | 매우 좋음 (한국어 최적화) | 무료 (셀프 호스팅) |
| `KoSimCSE` (오픈) | 768 | 한국어만 | 무료 |


> `dimensions` 옵션은 `text-embedding-3` 계열에서 지원. 차원을 줄이면 저장·검색 메모리는 줄지만 품질이 살짝 떨어질 수 있음.

비용·품질 trade-off. 한국어 강의 자료는 `text-embedding-3-small` 로 충분.


## 6. 정리

- RAG = Retrieve (외부 검색) + Augment (프롬프트 보강) + Generate (LLM 응답)
- 텍스트 → 벡터 → 코사인 유사도 비교
- 다음 노트북부터 5단계를 하나씩 구현

## [실습]
1. `embeddings.embed_query("카드값 증빙은 언제 올려야 해?")` 와 위 docs 의 유사도 확인.
2. text-embedding-3-large 로 바꿔 같은 비교 / 점수 차이?
3. `dimensions=512` 옵션으로 임베딩 차원 줄이고 결과 비교 (저장 비용 절감 효과).



In [8]:
question1 = '카드값 증빙은 언제 올려야해?'
q_vector1 = embeddings.embed_query(question1)

# 질문 vs 각 문서 유사도
scores = [cosine_sim(q_vector1, dv) for dv in doc_vectors]
ranked = sorted(zip(scores, docs), reverse=True)

print(f"질문: {question1}\n")
print("=== 유사도 순위 ===")
for score, doc in ranked:
    print(f"  {score:.3f}  {doc}")

질문: 카드값 증빙은 언제 올려야해?

=== 유사도 순위 ===
  0.482  법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다.
  0.234  재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.
  0.230  장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비스 복구 후 24시간 이내에 작성해야 한다.
  0.162  개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연락처는 마스킹 대상에 포함된다.
  0.151  신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사내 시스템 접근 권한이 제한될 수 있다.


In [9]:
docs = [
    "신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사내 시스템 접근 권한이 제한될 수 있다.",
    "법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다.",
    "개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연락처는 마스킹 대상에 포함된다.",
    "장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비스 복구 후 24시간 이내에 작성해야 한다.",
    "재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.",
]

doc_vectors = embeddings.embed_documents(docs)

In [11]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
doc_vectors = embeddings.embed_documents(docs)

vectors = embeddings.embed_documents(sentences)
print(f'벡터 차원: {len(vectors[0])}')

question1 = '카드값 증빙은 언제 올려야해?'
q_vector1 = embeddings.embed_query(question1)

def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# 질문 vs 각 문서 유사도
scores = [cosine_sim(q_vector1, dv) for dv in doc_vectors]
ranked = sorted(zip(scores, docs), reverse=True)

print(f"질문: {question1}\n")
print("=== 유사도 순위 ===")
for score, doc in ranked:
    print(f"  {score:.3f}  {doc}")

벡터 차원: 3072
질문: 카드값 증빙은 언제 올려야해?

=== 유사도 순위 ===
  0.485  법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다.
  0.264  장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비스 복구 후 24시간 이내에 작성해야 한다.
  0.217  개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연락처는 마스킹 대상에 포함된다.
  0.217  재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.
  0.216  신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사내 시스템 접근 권한이 제한될 수 있다.
